# 04. SurvFace 공식 probe 검색

공식 gallery의 동일 identity 전체를 평균한 `official_all` template 3,000개를 만들고, mated/unmated probe를 원래 `protocol_index` 순서로 top-20 검색합니다. 공식 test로 threshold/calibrator를 fit하지 않습니다.

| 범위 | 예상 시간 |
| --- | ---: |
| `EXECUTE_STAGE=False` | 1초 미만 |
| 역할별 `PROBE_LIMIT=100` smoke | 약 1~10분 |
| 전체 HNSW | 수시간 이상 |
| 전체 exact | 하루 이상 걸릴 수 있음 |

> **진행/체크포인트/재시작**: template materialization과 probe batch마다 진행률/heartbeat가 출력되어야 합니다. 중단되면 Kernel Restart 후 04 전체를 재실행합니다. 동일 `official_all` template은 provenance가 같으면 재사용하되, search mode/profile별 결과는 별도 attempt로 기록합니다. smoke 결과는 공식 지표로 사용하지 않습니다.


In [ ]:
# Step 1 실행 범위: 이 셀의 세 값만 바꾸고 Kernel Restart -> Run All
MODE = 'dev'             # 'dev' 또는 'real'
DATA_FRACTION = 1.0     # 0 < DATA_FRACTION <= 1
SEED = 42

import sys
from pathlib import Path

for _scope_root in (Path.cwd(), *Path.cwd().parents):
    if (_scope_root / 'research').is_dir():
        break
else:
    raise FileNotFoundError('D:/ronbun 내부에서 노트북을 실행하십시오.')
if str(_scope_root) not in sys.path:
    sys.path.insert(0, str(_scope_root))

from research.compression import PCA_SWEEP_DIMENSIONS
from research.experiments.scope import ExperimentScope

PCA_DIMENSIONS = (384, 256, 128, 64, 32)
PQ_SOURCE_DIMENSION = 512
if PCA_DIMENSIONS != tuple(PCA_SWEEP_DIMENSIONS):
    raise RuntimeError('노트북 PCA sweep과 공통 압축 정의가 다릅니다.')
EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)
EXPERIMENT_SCOPE.as_dict()


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = True
COMPRESSION_PROFILE = os.environ.get("RONBUN_SURVFACE_PROFILE", "origin_512")
SEARCH_MODE = os.environ.get("RONBUN_SURVFACE_SEARCH_MODE", "hnsw")
TOP_K = 20
PROBE_LIMIT = 100  # 역할별 smoke: 100, 공식 전체: None
BATCH_SIZE = 256
FULL_RUN_ACKNOWLEDGEMENT = ""  # 전체 실행 시 정확히 SURVFACE_FULL_SEARCH 입력
RUN_ROOT = PROJECT_ROOT / "runs" / "survface"

def resolve_run_for_preflight():
    try:
        return resolve_active_run(
            RUN_ROOT, environment_variable="RONBUN_SURVFACE_RUN_DIR"
        ), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

RUN_DIR, RUN_RESOLUTION_ERROR = resolve_run_for_preflight()
PROGRESS = ProgressReporter("SurvFace 04 official search", heartbeat_seconds=30)


## 1. 검색 계약 preflight

SurvFace 공식 MATLAB은 gallery identity별 평균 template와 rank 20을 사용합니다. 이 구현은 PostgreSQL/pgvector cosine distance 적응 실험이므로 원본 MATLAB의 Euclidean 결과와 bit-equivalent라고 표시하지 않습니다.


In [ ]:
preflight = {
    "execute_stage": EXECUTE_STAGE,
    "run_dir": str(RUN_DIR) if RUN_DIR else None,
    "run_resolution_error": RUN_RESOLUTION_ERROR,
    "compression_profile": COMPRESSION_PROFILE,
    "search_mode": SEARCH_MODE,
    "top_k": TOP_K,
    "probe_limit_per_role": PROBE_LIMIT,
    "gallery_enrollment_policy": "official_all",
    "gallery_enrollment_target": 0,
    "required_api": "research.experiments.run_survface_official_search",
}
preflight


## 2. official-all template 및 top-20 검색

필요 API가 없으면 명시적으로 중단합니다. API는 결과 CSV에 `probe_type`, `protocol_index`, `query_identity_id`, JSON 배열 `ranked_identities`/`ranked_distances`, profile/mode를 기록해야 합니다.


In [ ]:
result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    if RUN_DIR is None:
        raise RuntimeError(f"SurvFace run을 찾지 못했습니다: {RUN_RESOLUTION_ERROR}")
    if TOP_K != 20:
        raise ValueError("SurvFace 공식 평가용 top_k는 20이어야 합니다.")
    if COMPRESSION_PROFILE not in {"origin_512", "pca_256"}:
        raise ValueError("pgvector 검색 profile은 origin_512 또는 pca_256만 허용합니다.")
    if SEARCH_MODE not in {"exact", "hnsw"}:
        raise ValueError("SEARCH_MODE은 exact 또는 hnsw여야 합니다.")
    if PROBE_LIMIT is None and FULL_RUN_ACKNOWLEDGEMENT != "SURVFACE_FULL_SEARCH":
        raise RuntimeError("전체 검색 전 FULL_RUN_ACKNOWLEDGEMENT를 정확히 입력하십시오.")

    import pandas as pd
    import research.experiments as experiment_api

    from research.database import create_database_engine, init_database, load_database_settings
    from research.protocols import build_survface_official_protocol

    search_runner = getattr(experiment_api, "run_survface_official_search", None)
    if search_runner is None:
        raise NotImplementedError(
            "official_all template과 protocol_index 보존을 강제하는 공통 search API가 아직 없습니다. "
            "일반 LFW 검색 CSV를 SurvFace 공식 결과로 재사용하지 마십시오."
        )

    run = RunStore.open(RUN_DIR)
    run.verify_inputs()
    run.verify_phase_artifacts("01_official_arcface_embedding_extraction")
    if COMPRESSION_PROFILE == "pca_256":
        run.verify_phase_artifacts("03_official_compressed_materialization_and_index")
    manifest = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "survface" / "official_manifest.csv")
    protocol = build_survface_official_protocol(manifest)
    if not protocol.known_unknown_probes.empty:
        raise ValueError("SurvFace 공식 검색에 known_unknown이 들어갔습니다.")
    expected_counts = {
        "registered": len(protocol.registered_probes),
        "unknown_unknown": len(protocol.unknown_unknown_probes),
    }
    engine = create_database_engine(load_database_settings())
    init_database(engine)

    with run.phase("04_official_probe_search") as phase:
        suffix = f"A{phase.attempt:03d}"
        output_path = phase.attempt_dir / f"official_top20_{COMPRESSION_PROFILE}_{SEARCH_MODE}_{suffix}.csv"

        def report(message: str, details: dict[str, object]) -> None:
            PROGRESS.emit(message, **details)

        summary = search_runner(
            engine,
            run_uid=run.run_id,
            manifest=manifest,
            compression_profile=COMPRESSION_PROFILE,
            search_mode=SEARCH_MODE,
            top_k=TOP_K,
            enrollment_policy="official_all",
            enrollment_target=0,
            output_path=output_path,
            batch_size=BATCH_SIZE,
            probe_limit_per_role=PROBE_LIMIT,
            progress=report,
        )
        frame = pd.read_csv(output_path)
        required = {
            "probe_type", "protocol_index", "query_identity_id",
            "ranked_identities", "ranked_distances", "compression_profile", "search_mode",
        }
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(f"search 결과 필수 열 누락: {sorted(missing)}")
        if set(frame["probe_type"].astype(str)).difference({"registered", "unknown_unknown"}):
            raise ValueError("공식 결과에 known_unknown 또는 알 수 없는 probe_type이 있습니다.")
        for probe_type, group in frame.groupby("probe_type", sort=False):
            if not group["protocol_index"].astype(int).is_monotonic_increasing:
                raise ValueError(f"{probe_type} protocol_index 순서가 바뀌었습니다.")
            lengths = group["ranked_identities"].map(lambda value: len(json.loads(value)))
            if not lengths.eq(20).all():
                raise ValueError(f"{probe_type} 결과가 모두 top-20을 포함하지 않습니다.")

        actual_counts = frame["probe_type"].value_counts().to_dict()
        official_complete = (
            PROBE_LIMIT is None
            and actual_counts.get("registered", 0) == expected_counts["registered"]
            and actual_counts.get("unknown_unknown", 0) == expected_counts["unknown_unknown"]
        )
        frame["official_complete"] = official_complete
        frame["metric_label"] = "qmul-survface-v1-official-order-cosine-pgvector-adaptation"
        frame.to_csv(output_path, index=False, encoding="utf-8", lineterminator="\n")
        summary.update({
            "expected_counts": expected_counts,
            "actual_counts": actual_counts,
            "official_complete": official_complete,
            "known_unknown_count": 0,
            "top_k": 20,
            "gallery_enrollment_policy": "official_all",
            "gallery_enrollment_target": 0,
            "distance_semantics": "pgvector_cosine_distance",
        })
        summary_path = phase.attempt_dir / f"official_search_summary_{suffix}.json"
        summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        phase.publish_artifact(output_path)
        phase.publish_artifact(summary_path)
        phase.record_counts(rows=len(frame), **actual_counts)
    PROGRESS.emit("04 완료", rows=len(frame), official_complete=official_complete)
    result = {"status": "completed", "run_id": run.run_id, **summary}
else:
    PROGRESS.emit("검토 모드 완료: template/search를 실행하지 않음", expected="1초 미만")
result


## 다음 단계

논문용 공식-order 결과는 `official_complete=True`, known unknown 0, 각 probe top-20이어야 합니다. HNSW 결과는 같은 profile의 exact 결과와 recall/latency를 별도로 비교하고, official test score로 calibrator를 fit하지 않습니다.
